# Project 9: Market Basket — Association Rules

**Dataset:** Online Retail (UCI, 5,000 real transactions, 20 products)

**Goal:** Find products that are frequently bought together using Association Rule Learning (Apriori algorithm).

In [ ]:
# ====== Setup ======
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
DATA_URL = 'https://raw.githubusercontent.com/lucyyuhongxia-gmail/A4_ML_Projects/main/datasets'
def load_data(fn):
    df = pd.read_csv(f'{DATA_URL}/{fn}')
    print(f'Loaded: {fn} -> {df.shape[0]:,} rows x {df.shape[1]} cols')
    return df
print('Setup OK!')
from collections import Counter

## 1. Explore Transaction Data

In [ ]:
df = load_data('online_retail.csv')
print(f'Transactions: {df.InvoiceNo.nunique():,}')
print(f'Products: {df.Description.nunique()}')
print(f'Total rows: {len(df):,}')

# Group items by invoice — 按购物篮分组
transactions = df.groupby('InvoiceNo')['Description'].apply(list)
basket_sizes = transactions.apply(len)
print(f'\nBasket size stats:')
print(f'  Mean: {basket_sizes.mean():.1f} items')
print(f'  Range: {basket_sizes.min()} - {basket_sizes.max()}')

## 2. Find Frequent Items & Calculate Lift

In [ ]:
# Find items in >= 2% of baskets
all_items = [item for basket in transactions for item in basket]
item_counts = Counter(all_items)
n_baskets = len(transactions)

min_support = 0.02
frequent = {i: c/n_baskets for i,c in item_counts.items() if c/n_baskets >= min_support}
print(f'Frequent items (support >= {min_support}): {len(frequent)}')

# Calculate Lift for item pairs
# Lift = P(A&B) / (P(A) * P(B))
# Lift > 1 = positive association, Lift = 1 = independent
top_items = list(frequent.keys())[:10]
print(f'\n{"Item A":35s} {"Item B":35s} {"Support":8s} {"Lift":6s}')
print('-' * 85)
for a in top_items:
    for b in top_items:
        if a >= b: continue
        both = sum(1 for basket in transactions if a in basket and b in basket)
        if both == 0: continue
        p_ab = both/n_baskets; p_a = item_counts[a]/n_baskets; p_b = item_counts[b]/n_baskets
        lift = p_ab / (p_a * p_b)
        if lift > 1.5:
            print(f'{a[:34]:35s} {b[:34]:35s} {p_ab:8.4f} {lift:6.2f}')

## Check Your Understanding
1. Lift=2.32 for "TEA TOWELS 2pk → TEA TOWELS 3pk". Explain this in plain English.
2. If Lift=0.8, what would it mean? Would you place these items next to each other?
3. Based on "CAKESTAND → T-LIGHT HOLDER" (Lift=2.17), what business action would you take?
4. What's the problem with having too many rules? How does Apriori's pruning help?